# The yes/no task: one sound, one answer

This is the newer of the two SeqSFG tasks. The listener hears **one** sound and says whether a
figure was there. No comparison, no second interval.

That sounds like the easier task to build. It is the harder one, and this notebook is mostly
about why — and about the version of it that I threw away after measuring it.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/notebooks/SeqSFG_yesno_playground.ipynb)

**Runtime → Run all**, CPU is fine, takes about two minutes. Nothing autoplays. Use headphones
at a comfortable volume and don't change it partway through — a couple of the demonstrations
are about level, and they only make sense if yours is fixed.

Everything below is computed live from the repository at commit `310b2fc7ebb3`. There are no
numbers pasted in from elsewhere; if you change the config and re-run, the conclusions move.

In [ ]:
#@title Setup: clone the repository and load the shipped configuration
import sys, os, subprocess, importlib.util, json
from pathlib import Path
SOURCE_REF='310b2fc7ebb3661c3f0bc70e98738842fd5aab05'
REPO_URL='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
IN_COLAB=bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
if IN_COLAB:
    ROOT=Path('/content')/('seqsfg-yesno-'+SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/yesno.py').exists()),None)
    if ROOT is None: raise RuntimeError('Run from inside the repository, or open in Colab.')
missing=[m for m in ['numpy','scipy','matplotlib','ipywidgets'] if importlib.util.find_spec(m) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
sys.path.insert(0,str(ROOT))

import numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
from seqsfg.config import Config, validate
from seqsfg.stimulus import render_interval, FIGURE
from seqsfg import yesno, measure as M, verify as V

cfg=Config.from_dict(json.loads((ROOT/'pilot_config.json').read_text()))
D=validate(cfg); SR=cfg.sample_rate
REF_LEVEL=M.single_tone_reference(cfg,D,V.WIN_MS,V.HOP_MS)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.25,'font.size':9})
INK={'present':'#c1272d','absent':'#1b6ca8','plain':'#e08c1a','scattered':'#6b3fa0','bg':'#b9bec6'}

def sound(seed, present, absent='roving', step=0.0):
    iv=yesno.build_interval(cfg,D,seed,step,present,absent)
    return iv, render_interval(cfg,iv,D)

def play(x, gain=1.0):
    return Audio(np.clip(x*gain,-1,1), rate=SR, normalize=False)

print('commit      ', SOURCE_REF[:12])
print('config      ', cfg.hash())
print(f'{D.n_channels} channels, {cfg.pool_low_hz:.0f}-{D.channel_freqs_hz[-1]:.0f} Hz, 1 ERB apart')
print(f'{cfg.n_elements} elements of {cfg.n_components} components at '
      f'{1000/cfg.iei_max_ms:.1f}-{1000/cfg.iei_min_ms:.1f} Hz over {cfg.interval_dur_ms/1000:g} s')
print(f'{cfg.tone_dur_ms:g} ms tones, {cfg.ramp_ms:g} ms ramps, {cfg.tones_per_channel} tones per channel')
print(f'elements banded to {cfg.figure_band_channels} channels; ladder {list(cfg.steps_ms)} ms')

---
## 1. Listen before you read anything else

Two sounds. One of them contains a figure — seven tones that start together and come back on
the *same* pitches, nine times, about three times a second. The other does not.

I have deliberately not told you which is which yet. Play them a couple of times each.

In [ ]:
a_iv,a_x = sound(seed=101, present=True)
b_iv,b_x = sound(seed=202, present=False)
display(Markdown('**Sound A**')); display(play(a_x))
display(Markdown('**Sound B**')); display(play(b_x))

If A felt like it had a repeating "chord" in it and B felt like it was wandering, you heard it:
**A is present, B is absent.**

If they both sounded like a wash of pips, that is also a completely normal first reaction. The
figure is seven tones out of about nine sounding at any instant, and it only becomes an object
after a few repetitions. Try A again knowing what to listen for.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
oct_axis=np.log2(D.channel_freqs_hz)
for ax,(iv,title) in zip(axes,[(a_iv,'A — figure present'),(b_iv,'B — figure absent')]):
    t=iv.onset*cfg.grid_ms/1000.0; y=oct_axis[iv.channel]
    fig_m=iv.kind==FIGURE
    ax.scatter(t[~fig_m],y[~fig_m],s=6,c=INK['bg'],marker='s',linewidths=0)
    ax.scatter(t[fig_m],y[fig_m],s=22,c=INK['present'] if 'present' in title else INK['absent'],
               marker='s',linewidths=0)
    ax.set_title(title); ax.set_xlabel('time (s)')
    ax.set_xlim(0,cfg.interval_dur_ms/1000)
axes[0].set_ylabel('frequency (octaves re 1 Hz)')
ticks=[250,500,1000,2000,4000,8000]
axes[0].set_yticks(np.log2(ticks)); axes[0].set_yticklabels([str(t) for t in ticks])
axes[0].set_ylabel('frequency (Hz)')
fig.suptitle('grey = background   coloured = the tones that start together',y=1.0)
plt.tight_layout(); plt.show()

## 2. What the picture shows

The coloured squares are the tones that start together — one group per element, nine of them,
in **both** panels. That is the whole trick of this design and it is worth pausing on.

- On the left the group returns to the **same band of pitches** every time. That is the figure.
- On the right the group is just as tight, just as loud, and just as frequent — but it **lands
  somewhere else each time**. Nothing recurs.

So "absent" here does not mean "no group". It means "no group that comes back". I did not
choose that out of taste; I chose it after measuring the alternative, which is section 3.

---
## 3. The version I threw away

The obvious way to build a yes/no figure-detection task is: figure versus a plain cloud with no
figure at all. That is the classic stochastic-figure-ground detection task and it is what I
tried first.

Listen to a plain cloud next to a present trial.

In [ ]:
p_iv,p_x = sound(seed=303, present=True)
c_iv,c_x = sound(seed=404, present=False, absent='plain')
display(Markdown('**Figure present**')); display(play(p_x))
display(Markdown('**Plain cloud (the naive "absent")**')); display(play(c_x))

You can probably hear the difference immediately, which feels like good news. It is not.

The question is never "can a listener tell these apart" — it is "**can they tell them apart
without hearing a figure**". Here is the broadband envelope of each. Same tones, same count,
same long-term spectrum. Look at what synchrony does to the shape.

In [ ]:
def env(x): return M.frame_rms(x, SR, 2.0)
fig,axes=plt.subplots(2,1,figsize=(12,4.6),sharex=True,sharey=True)
for ax,(x,lab,col) in zip(axes,[(p_x,'figure present',INK['present']),
                                (c_x,'plain cloud',INK['plain'])]):
    e=env(x); t=np.arange(e.size)*2.0/1000.0
    ax.plot(t,e/e.mean(),lw=0.8,color=col)
    ax.axhline(1,color='k',lw=0.5,alpha=0.4); ax.set_ylabel(lab+'\nenvelope / mean')
axes[-1].set_xlabel('time (s)'); axes[0].set_title('broadband envelope, 2 ms frames')
plt.tight_layout(); plt.show()

The present trial has a spike every ~316 ms. Nine tones-worth of energy arriving in one instant
instead of spread across the element is a level event, and it is periodic. You do not need to
hear a figure to notice a rhythm.

The autocorrelation makes it unmissable.

In [ ]:
def env_ac(x, max_lag_ms=1200):
    e=env(x); dev=e/e.mean()-1.0
    ac=np.correlate(dev,dev,'full')[dev.size-1:]; ac/=ac[0]
    n=int(max_lag_ms/2.0); return np.arange(n)*2.0, ac[:n]
plt.figure(figsize=(11,3.4))
for x,lab,col in [(p_x,'figure present',INK['present']),(c_x,'plain cloud',INK['plain']),
                  (a_x,'absent, roving (section 1)',INK['absent'])]:
    lag,ac=env_ac(x); plt.plot(lag,ac,lw=1.3,label=lab,color=col)
plt.axvspan(cfg.iei_min_ms,cfg.iei_max_ms,color='k',alpha=0.07)
plt.text(np.mean([cfg.iei_min_ms,cfg.iei_max_ms]),0.55,'element rate',ha='center',fontsize=8)
plt.xlabel('lag (ms)'); plt.ylabel('envelope autocorrelation'); plt.legend(frameon=False)
plt.title('a chord every 316 ms is a rhythm, and a rhythm is audible without hearing a figure')
plt.tight_layout(); plt.show()

The orange curve peaks in the element-rate band. The red one peaks there too — of course, it
also has a chord every element. The blue one, the absent class I actually use, sits on top of
the red one.

That single feature (`env:ac_peak_iei`) is the one the full audit flags hardest for the plain
cloud, at **d′ = 1.79**. Let me make the consequence concrete rather than abstract: below is a
detector that has never heard of a figure. It measures one number per sound and says "yes" if
the number is above a threshold. Drag the threshold and watch what it can do.

In [ ]:
def feature_of(x, name='env:ac_peak_iei', iv=None, step=0.0):
    span=(cfg.n_components-1)*step+cfg.figure_repeats*cfg.tone_dur_ms
    m=V.measure_interval(cfg,D,iv,x,iv.figure_set,span,REF_LEVEL)
    return V.scalar_features(m)[name]

N=40
vals={}
for kind,lab in [('plain','plain cloud'),('roving','roving (shipped)')]:
    pres=[];abst=[]
    for j in range(N):
        ivp,xp=sound(seed=1000+j,present=True); ivq,xq=sound(seed=5000+j,present=False,absent=kind)
        pres.append(feature_of(xp,iv=ivp)); abst.append(feature_of(xq,iv=ivq))
    vals[kind]=(np.array(pres),np.array(abst),lab)

fig,axes=plt.subplots(1,2,figsize=(12,3.6))
for ax,kind in zip(axes,['plain','roving']):
    pres,abst,lab=vals[kind]
    bins=np.linspace(min(pres.min(),abst.min()),max(pres.max(),abst.max()),26)
    ax.hist(abst,bins=bins,alpha=0.65,label='absent',color=INK['plain'] if kind=='plain' else INK['absent'])
    ax.hist(pres,bins=bins,alpha=0.65,label='present',color=INK['present'])
    auc=(pres[:,None]>abst[None,:]).mean()+0.5*(pres[:,None]==abst[None,:]).mean()
    ax.set_title(f"{lab}\nAUC={auc:.3f}   d'={float(yesno.auc_dprime(np.array(auc))):+.2f}")
    ax.set_xlabel('envelope autocorrelation at element lags'); ax.legend(frameon=False,fontsize=8)
axes[0].set_ylabel(f'trials (n={N} each)')
plt.tight_layout(); plt.show()

Two distributions that barely touch on the left; two that sit on top of each other on the right.
The left panel is a task a machine can do perfectly while being completely deaf to grouping.

**This is the whole reason the yes/no task needed its own audit.** In the two-interval task the
listener compares two sounds and everything common to both cancels. Here they answer from one
sound against their memory of what these sounds are usually like, so any property whose
*distribution* differs — in mean **or in spread** — is a usable criterion.

---
## 4. Three candidate "absent" classes, and the one that survived

- **`plain`** — a cloud with no element structure at all. Classic SFG detection.
- **`scattered`** — the same channels at the same element times, but the components scattered
  inside the element so nothing binds. Recurrence without binding.
- **`roving`** — elements bound and just as frequent, but landing on a fresh band every time,
  so nothing recurs.

Hear all three against a present trial, then look at the numbers.

In [ ]:
display(Markdown('**present**')); display(play(sound(seed=77,present=True)[1]))
for kind in yesno.ABSENT_KINDS:
    display(Markdown(f'**absent — `{kind}`**'))
    display(play(sound(seed=88,present=False,absent=kind)[1]))

Now the audit. This is a live run, not a stored table: 25 present and 25 absent trials at step 0
for each class, all 65 measurable features, with a max-statistic permutation test so that
testing 65 correlated features does not manufacture significance.

It takes about a minute.

In [ ]:
rows=[]
for kind in yesno.ABSENT_KINDS:
    r=yesno.run_audit(cfg,n_trials=25,seed=99,steps=[0.0],absent=kind,n_perm=4000,verbose=False)
    s=r['pooled']; top=int(np.argmax(s['dprime']))
    rows.append((kind,s['p_value'],s['dprime'][top],s['names'][top],
                 r['pooled_learnt']['dprime'],r['pooled_learnt']['pc']))
display(Markdown('| absent class | permutation p | largest feature d′ | which feature | learnt observer d′ | % correct |\n'
                 '|---|---|---|---|---|---|\n' + '\n'.join(
    f"| `{k}` | {p:.4f} | {d:.2f} | `{n}` | {ld:+.2f} | {pc*100:.0f}% |" for k,p,d,n,ld,pc in rows)))

The last two columns are the ones that matter. That observer is a ridge-logistic classifier with
leave-one-out cross-validation and access to every feature at once — an upper bound on what any
listener could learn about "what these sounds are usually like" over a session. Against `plain`
it is near ceiling. Against `roving` it is at chance, which is what you want and what you rarely
get for free.

(Your numbers will wobble a little from the ones in the README, which used 60 trials at each of
six steps rather than 25 at one. The ordering does not wobble.)

---
## 5. The envelope, under a microscope

You told me to be careful about bursts, so here is the part I would want to see if I were
reviewing this. Four properties a listener could plausibly key on, plotted as distributions
rather than means — because in yes/no a difference in **spread** is a criterion too.

In [ ]:
FEATS=[('rms_db','long-term RMS (dB)'),('env:n_bursts_3sd','envelope peaks above 3 SD'),
       ('env:crest','crest factor'),('env:kurtosis','envelope kurtosis')]
N2=45
def features_for(present,kind,n,seed0):
    out=[]
    for j in range(n):
        iv,x=sound(seed=seed0+j,present=present,absent=kind)
        span=cfg.figure_repeats*cfg.tone_dur_ms
        out.append(V.scalar_features(V.measure_interval(cfg,D,iv,x,iv.figure_set,span,REF_LEVEL)))
    return out
Fp=features_for(True,'roving',N2,20000)
Fa=features_for(False,'roving',N2,30000)
fig,axes=plt.subplots(1,4,figsize=(14,3.1))
for ax,(key,lab) in zip(axes,FEATS):
    p=np.array([f[key] for f in Fp]); a=np.array([f[key] for f in Fa])
    bins=np.histogram_bin_edges(np.r_[p,a],bins=18)
    ax.hist(a,bins=bins,alpha=0.6,color=INK['absent'],label='absent')
    ax.hist(p,bins=bins,alpha=0.6,color=INK['present'],label='present')
    auc=(p[:,None]>a[None,:]).mean()+0.5*(p[:,None]==a[None,:]).mean()
    ax.set_title(f"{lab}\nd'={float(yesno.auc_dprime(np.array(auc))):+.2f}",fontsize=9)
axes[0].legend(frameon=False,fontsize=8); axes[0].set_ylabel(f'trials (n={N2})')
plt.tight_layout(); plt.show()

Overlapping, and overlapping to the same width. The long-term RMS one is not even a
distribution worth plotting, and that is by construction rather than by luck: **every interval
of either class carries exactly the same number of tones in exactly the same channels.** Total
tone count is not a random variable in this design. There is a test that asserts it.

---
## 6. Try it yourself

Sixteen trials, half present, half absent, in a random order I am not showing you. Answer each
one, then run the scoring cell. It reports d′ and criterion c, not percent correct — in yes/no
the two are different things, because you choose your own threshold for saying "yes".

If the buttons do not render (some Jupyter setups), the fallback cell below scores a list you
type by hand.

In [ ]:
rng=np.random.default_rng(2026)
TRUTH=list(rng.permutation([True]*8+[False]*8))
SEEDS=[int(s) for s in rng.integers(1,2**31-1,size=16)]
ANSWERS={}
try:
    import ipywidgets as W
    rows=[]
    for i,(t,s) in enumerate(zip(TRUTH,SEEDS)):
        _,x=sound(seed=s,present=bool(t))
        btn=W.ToggleButtons(options=[('—',None),('yes','y'),('no','n')],value=None,
                            description=f'{i+1:2d}.',layout=W.Layout(width='300px'))
        ANSWERS[i]=btn
        rows.append(W.HBox([btn,W.Output()]))
        with rows[-1].children[1]: display(play(x))
    display(W.VBox(rows))
except Exception as e:
    print('widgets unavailable:',e)
    for i,(t,s) in enumerate(zip(TRUTH,SEEDS)):
        _,x=sound(seed=s,present=bool(t))
        display(Markdown(f'**trial {i+1}**')); display(play(x))

In [ ]:
#@title Score me
MY_ANSWERS=None   # or paste e.g. 'ynyynnyn ynynynyn' if the buttons did not render
if MY_ANSWERS is None and len(ANSWERS)==16:
    got=[ANSWERS[i].value for i in range(16)]
elif MY_ANSWERS:
    got=[c for c in MY_ANSWERS if c in 'yn']
else:
    got=[]
if any(g is None for g in got) or len(got)!=16:
    print('answer all sixteen first (or set MY_ANSWERS).')
else:
    h=sum(1 for g,t in zip(got,TRUTH) if t and g=='y'); ns=sum(TRUTH)
    f=sum(1 for g,t in zip(got,TRUTH) if not t and g=='y'); nn=16-ns
    dp=yesno.dprime_yesno(h/ns,f/nn,ns,nn); c=yesno.criterion_yesno(h/ns,f/nn,ns,nn)
    print(f'hits {h}/{ns}   false alarms {f}/{nn}')
    print(f"d' = {dp:+.2f}    criterion c = {c:+.2f} "
          f"({'you leaned towards yes' if c<-0.25 else 'you leaned towards no' if c>0.25 else 'roughly unbiased'})")
    print(f'percent correct {(h+(nn-f))/16*100:.0f}%  <- not the measure, but people always want it')
    print('\nsixteen trials is far too few to estimate d\' well; the point is the shape of the answer.')

---
## 7. Break it on purpose

The audit is only worth trusting if it catches things. Here I plant a cue that no listener would
consciously notice — present trials made **0.1 dB** louder — and check that the audit refuses it.

Then, for contrast, a difference in spread only: same mean level, more variable across trials.
That is the one people forget, and it is just as usable as a mean shift.

In [ ]:
names=list(Fp[0].keys())
Ap=np.array([[f[k] for k in names] for f in Fp]); Aa=np.array([[f[k] for k in names] for f in Fa])
base=yesno.feature_separation(names,Ap,Aa,n_perm=4000,seed=3)

shift=Ap.copy(); shift[:,names.index('rms_db')]+=0.1
r_shift=yesno.feature_separation(names,shift,Aa,n_perm=4000,seed=3)

spread=Ap.copy(); j=names.index('env:crest')
col=spread[:,j]; spread[:,j]=col.mean()+(col-col.mean())*2.5
r_spread=yesno.feature_separation(names,spread,Aa,n_perm=4000,seed=3)

for lab,r in [('as shipped',base),('present trials +0.1 dB',r_shift),
              ('crest factor 2.5x more variable, same mean',r_spread)]:
    top=int(np.argmax(r['dprime']))
    print(f"{lab:<44} p={r['p_value']:.4f}   largest d'={r['dprime'][top]:.2f} "
          f"({r['names'][top]}, {r['kind'][top]})")

A tenth of a decibel is well under anything a listener would call a loudness difference, and the
audit still rejects. The spread-only sabotage is caught too, and labelled `spread` rather than
`shift`, which is the distinction that makes this audit different from the two-interval one.

If you want to play: change `absent='roving'` to `'scattered'` anywhere above and re-run
section 4. You will watch a design fail its own test.

---
## 8. Where this leaves you

What I would actually claim from this notebook:

1. **The naive yes/no task does not work.** Figure versus plain cloud is 90% solvable from the
   envelope alone. If you see that task in a paper without an envelope control, be suspicious.
2. **`roving` is a real fix, not a patch.** Both classes contain the same number of bound
   chords, so the envelope matches by construction rather than by tuning.
3. **It changes the question.** "Is a figure present?" becomes "did *one* figure keep coming
   back?". That is the single-interval form of the two-interval task, and it is a narrower claim
   than figure detection. I would write it that way in a paper rather than hoping nobody asks.

What I would not claim:

- That `roving` is confound-free in general. It is clean on 65 measured features at this
  configuration, at these trial counts. Change the config and re-run the audit; that is what it
  is for.
- That the listener's d′ here means the same thing as d′ in a classic SFG detection experiment.
  It does not, and the paragraph above says why.

```
seqsfg yesno-verify --config pilot_config.json --absent all
seqsfg yesno-run    --config pilot_config.json --code P01
seqsfg yesno-analyze data/P01/session_01
```